Here we will build the baseline model. In this case we will design a simple allocator with fixed weight

Our simple allocator will then be the benchmark to beat

# Imports, data and variables definition

In [22]:
# imports

import pandas as pd
import numpy as np 
from vportfolio.params import RAW, PROCESSED, LOOKBACK, CAP

In [10]:
# load the data

raw_data    = pd.read_parquet(RAW / "yahoo_prices_2026-10-23.parquet")
rebal_dates = pd.read_parquet(PROCESSED / "rebal_dates.parquet")["date"]
cost_bp     = pd.read_parquet(PROCESSED / "cost_bp.parquet")["cost_bp"]
rf_daily    = pd.read_parquet(PROCESSED / "rf_daily.parquet")["rf"]

In [11]:
# variables definition

b = ["SPY", "AGG"]
close_b = raw_data["Close"][b]
open_b  = raw_data["Open"][b]
returns_b = close_b.pct_change(fill_method=None)

# Helper Functions

In [12]:
# along the lines of the last test function:
def lookback_window(returns, eligible, date, n=LOOKBACK):
    assets = eligible.columns[eligible.loc[date]] # which ETFs are available on a given date by TRUE/FALSE
    window = returns.loc[:date, assets].tail(n) # lookback period of 504 raws (2 years) from the given date
    assert window.notna().all().all() # double checks and stops if lookback period contains NaN (extra safety after data tests)
    return window

In [79]:
# function testing

dates = pd.bdate_range("2020-1-1", periods=600) # a bit more than 504 days
fake_returns = pd.DataFrame(0.001, index=dates, columns=["A", "B", "C", "D", "E"]) # constant returns
fake_returns.loc[:dates[200], "E"] = np.nan # E starts trading later
fake_eligible = fake_returns.notna().cumsum() >= LOOKBACK # same rule as preprocessing
check_day = dates[-1] # last date: A-D eligible, E not yet

window = lookback_window(fake_returns, fake_eligible, check_day)
assert len(window) == LOOKBACK # exactly 504 rows
assert window.index[-1] == check_day # ends on the requested date: no look-ahead
assert "E" not in window.columns # E not eligible yet, so excluded
print("lookback_window checks passed")

lookback_window checks passed


In [78]:
# apply the maximum weight cap of 20% maximum per ETF
def apply_cap(w, cap=CAP):
    w = w.copy()
    for _ in range(100): # repeat the capping step until nothing is over the cap but never more than 100 times
        # for loop is necssary as redistributing can generate weights above 20% so needs to continue until all are in line
        over = w > cap + 1e-12 # finds weights above cap of 20% + rounding error of small decimals
        if not over.any(): 
            break # exits the loop when no weights are over the cap
        excess = (w[over] - cap).sum() # calculates the excess of over - cap as a total
        w[over] = cap # cuts those weights to CAP
        under = ~over & (w < cap) # finds weights below cap
        w[under] += excess * w[under] / w[under].sum() # adds to the current values of the under capped ETFs weight a share of the excess
        # proportional to their weights
    return w

In [74]:
# function testing
w_equal = pd.Series({"A": 0.2, "B": 0.2, "C": 0.2, "D": 0.2, "E": 0.2}) # nothing above the cap
w_over = pd.Series({"A": 0.5, "B": 0.2, "C": 0.1, "D": 0.1, "E": 0.05, "F": 0.05}) # A above the cap
w_over_copy = w_over.copy() # to check the original is not modified

capped_equal = apply_cap(w_equal)
capped_over = apply_cap(w_over)

assert (capped_equal == w_equal).all() # 1. unchanged when nothing is over the cap
assert (capped_over <= 0.20 + 1e-12).all() # 2. no weight above the cap
assert abs(capped_over.sum() - 1) < 1e-12 # 2. weights still sum to 1
assert abs(capped_over["E"] / capped_over["F"] - 1) < 1e-12 # 3. E and F started equal, end equal
assert abs(capped_over["C"] / capped_over["E"] - 2) < 1e-12 # 3. C was twice E, still twice E
assert (w_over == w_over_copy).all() # 4. original weights not modified
print("apply_cap checks passed")
capped_over

apply_cap checks passed


A    0.2
B    0.2
C    0.2
D    0.2
E    0.1
F    0.1
dtype: float64

# Simple Model

The simple model will behave as follows

1. First rebalance date: decide target weights (60% SPY / 40% AGG) at the close
2. Next trading day: buy to the target weights at the open
3. Pay trading costs on the amount traded
4. Hold until month end; weights drift with prices
5. Last trading day of the month: decide target weights again (60/40) at the close
6. First trading day of the next month: trade back to 60/40 at the open
   (only the difference from the drifted weights), paying costs on that amount
7. Repeat steps 4-6 until the last rebalance date

The same steps will apply to every allocator, the main difference will be the weights used in step 1 and 5

To replicate this I will build separate functions to

1. decide target weights at month end close based on difference with allocator
2. estimate trading costs based on decision from step 1
3. estimate daily NAV from trade-day open to the day before next trade
4. estimate value and weights at the next trade day's open

In [ ]:
# step 1; decide

# applies allocator methodology to final target weight
# will return the final weights per ETF, and 0 for the unavailable ones

def new_target(allocator, returns, eligible, day, cap, lookback, tickers): # allocator passed as an argument
    w = allocator(lookback_window(returns, eligible, day, lookback)) # build lookback period for given day and give it to the allocator
    if cap is not None: 
        w = apply_cap(w, cap) # apply cap function unless is not set
    return w.reindex(tickers, fill_value=0.0)

In [149]:
# function testing
def half_half(window): # dummy allocator: 50/50 in A and B, ignores the data
    return pd.Series({"A": 0.5, "B": 0.5})

target = new_target(half_half, fake_returns, fake_eligible, check_day, None, LOOKBACK, fake_returns.columns)
assert list(target.index) == list(fake_returns.columns) # one entry for every ticker
assert (target[["C", "D", "E"]] == 0).all() # tickers not chosen get 0
assert abs(target.sum() - 1) < 1e-12 # weights sum to 1
print("new_target checks passed")

new_target checks passed


In [150]:
# step 2; trade

# the results, cost and turnover, will feed into every trade

def trading_costs(target, weights, cost_bp):
    traded = (target - weights).abs()
    turnover = traded.sum()
    cost = (traded * cost_bp/10000).sum()    
    return pd.Series({"cost": cost, "turnover": turnover})
    

In [151]:
# function testing
target = pd.Series({"A": 0.6, "B": 0.4})
weights1 = pd.Series({"A": 0.0, "B": 0.0}) # at start
weights2 = pd.Series({"A": 0.6, "B": 0.4}) # already at target
weights3 = pd.Series({"A": 0.62, "B": 0.38}) # drifted
cost_bp1 = pd.Series({"A": 10, "B": 10}) # 10 bp each
cost_bp2 = pd.Series({"A": 2, "B": 10}) # different costs

test1 = trading_costs(target, weights1, cost_bp1) # 100% traded at 0.1%
test2 = trading_costs(target, weights2, cost_bp1) # 0% traded
test3 = trading_costs(target,weights3, cost_bp1) # traded 4%
test4 = trading_costs(target,weights3, cost_bp2) # traded 4% at diferent costs

assert np.isclose(test1["cost"], 1*0.001,atol=1e-12)
assert np.isclose(test2["cost"], 0*0.001,atol=1e-12)
assert np.isclose(test3["cost"], 0.04*0.001, atol=1e-12)
assert np.isclose(test4["cost"], 0.02*0.0002 + 0.02*0.001, atol=1e-12)

assert np.isclose(test1["turnover"], 1, atol=1e-12)
assert np.isclose(test2["turnover"], 0, atol=1e-12)
assert np.isclose(test3["turnover"], 0.04, atol=1e-12)
assert np.isclose(test4["turnover"], 0.04, atol=1e-12)

print("trading_costs checks passed")

trading_costs checks passed


In [152]:
# step 3: hold

# portfolio NAV from trad_day to next_trade_day

def holding_nav(value, target, close_p, open_p, trade_day, next_trade_day):
    cp = close_p.loc[trade_day: next_trade_day].iloc[:-1] # take closing prices of holding period, excluding last one
    op = open_p.loc[trade_day] # take open prices at trading date
    etf_growth = cp/op # calculate growht
    new_nav = value * (target * etf_growth).fillna(0).sum(axis=1) # calculates change in protfolio $ value and assigns 0 tho NaNs
    return new_nav   
    

In [162]:
# function testing
dates = pd.bdate_range("2020-1-1", periods=6)

# test 1 : NAV flat
close_flat= pd.DataFrame({"A": [100] * 6, "B": [100] * 6}, index = dates) # all open and close set to 100
open_flat = close_flat.copy()
weights_equal = pd.Series({"A": 0.5, "B": 0.5}) # equal weights
trade_day = dates[0]
next_trade_day = dates[5]
test1= holding_nav(1.0,weights_equal, close_flat, open_flat, trade_day, nex_trade_day)
assert np.isclose(test1, 1.0, atol=1e-12).all() 

# test 2: NAV of 1 ETF, other stable
close_up = close_flat.copy()
close_up.loc[dates[2]:, "A"] = 200
open_up = open_flat.copy()
open_up.loc[dates[3]:, "A"] = 200
weights_1 = pd.Series({"A": 0.5, "B": 0.5}) # equal weights
test2= holding_nav(1.0,weights_1, close_up, oppen_up, trade_day, nex_trade_day)
assert np.isclose(test2.iloc[2:], 1.5, atol=1e-12).all()

# test 3: ETF with NANs
close_na = close_flat.copy()
close_na["C"] = np.nan # add ETF with NANs
open_na = close_na.copy()
weights_na = pd.Series({"A": 0.5, "B": 0.5, "C": 0.0}) # give C 0 weight
test3 = holding_nav(1.0, weights_na, close_na, open_na, trade_day, next_trade_day)
assert test3.notna().all() # no NaN in the NAV
assert np.isclose(test3, 1.0, atol=1e-12).all() # A and B flat: NAV unchanged

print("holding_nav checks passed")

holding_nav checks passed


In [163]:
# step 4: drift

# calculates portfolio value at next trade day's open and the new weights
def drift(value, target, open_price, trade_day, next_trade_day):
    ETF_growth = (open_price.loc[next_trade_day]/open_price.loc[trade_day]).fillna(0)
    relative_weight = target * ETF_growth
    new_nav = value*(relative_weight.sum())
    new_weights = relative_weight / (relative_weight).sum()
    
    return pd.Series({"NAV": new_nav, "Weights": new_weights})

In [164]:
# function testing

# we can reuse the elements from previous function's testing
test1 = drift(1.0, weights_equal, open_flat, trade_day, next_trade_day)
assert np.isclose(test1["NAV"], 1.0, atol=1e-12).all() 
assert np.isclose(test1["Weights"], [0.5, 0.5], atol=1e-12).all() 

test2 = drift(1.0, weights_equal, open_up, trade_day, next_trade_day)
assert np.isclose(test2["NAV"], 1.5, atol=1e-12)
assert np.isclose(test2["Weights"]["A"], 2/3, atol=1e-12)

test3 = drift(1.0, weights_na, open_na, trade_day, next_trade_day)
assert test3["Weights"].notna().all()
assert np.isclose(test3["Weights"], [0.5, 0.5, 0], atol=1e-12).all()

print("drift checks passed")

drift checks passed


We can now chain these functions into a single one, the backtesting engine, which will repeat the steps monthly

In [181]:
def backtest_engine (allocator, close_price, open_price, returns, eligible, rebal_dates, cost_bp, cap=CAP, lookback = LOOKBACK): 
    value = 1.0 # set starting value a 1$
    weights = pd.Series(0.0, index=close_price.columns) # set starting weights to 0
    monthly_NAV = [] # empty list to store NAVs
    turnover = {}

    for i, decision_day in enumerate(rebal_dates[:-1]):
        dates = close_price.index
        trade_day = dates[dates.get_loc(decision_day) + 1]
        next_trade_day = dates[dates.get_loc(rebal_dates.iloc[i + 1]) + 1]

        # step 1
        target = new_target(allocator, returns, eligible, decision_day, cap, lookback, close_price.columns)
        # step 2
        costs_turnover = trading_costs(target, weights, cost_bp)
        turnover[trade_day] = costs_turnover["turnover"]
        value = value * (1 - costs_turnover["cost"])
        # step 3
        daily_nav = holding_nav(value, target, close_price, open_price, trade_day, next_trade_day)
        monthly_NAV.append(daily_nav)
        # step 4
        nav_weights = drift(value, target, open_price, trade_day, next_trade_day)
        value = nav_weights["NAV"]
        weights = nav_weights["Weights"]
        
    return pd.Series({"NAV": pd.concat(monthly_NAV), "Turnover": pd.Series(turnover)})

In [182]:
# function testing

# fake market data
dates_long = pd.bdate_range("2020-1-1", periods=756) # about 3 years of business days
t = np.arange(len(dates_long))
close_fake = pd.DataFrame({"AAA": 100 * 1.001 ** t, "BBB": [50.0] * len(dates_long)}, index=dates_long) # AAA +0.1% a day, BBB flat
open_fake = close_fake.shift(1) # open = previous day's close
open_fake.iloc[0] = close_fake.iloc[0] # first day: open = close
returns_fake = close_fake.pct_change(fill_method=None)
eligible_fake = close_fake.notna() # everything eligible
month_days = returns_fake.loc["2020-03-01":"2022-10-31"].index # start a few months in
rebal_fake = month_days.to_series().groupby([month_days.year, month_days.month]).max()
cost_zero = pd.Series({"AAA": 0, "BBB": 0}) # no costs
cost_ten = pd.Series({"AAA": 10, "BBB": 10}) # 10 bp each

# dummy allocators
def all_in_aaa(window): # always 100% AAA
    return pd.Series({"AAA": 1.0})

def half_half(window): # always 50/50
    return pd.Series({"AAA": 0.5, "BBB": 0.5})

# test 1: buy and hold, no costs
test1 = backtest_engine(all_in_aaa, close_fake, open_fake, returns_fake, eligible_fake, rebal_fake, cost_zero, cap=None, lookback=1)
first_trade_day = test1["Turnover"].index[0]
last_nav_day = test1["NAV"].index[-1]
expected = close_fake.loc[last_nav_day, "AAA"] / open_fake.loc[first_trade_day, "AAA"]
assert np.isclose(test1["NAV"].iloc[-1], expected, atol=1e-12) # matches holding AAA directly

# test 2: first trade from cash is 100%
assert np.isclose(test1["Turnover"].iloc[0], 1.0, atol=1e-12)

# test 3: costs charged once at 10 bp
test3 = backtest_engine(all_in_aaa, close_fake, open_fake, returns_fake, eligible_fake, rebal_fake, cost_ten, cap=None, lookback=1)
assert np.isclose(test3["NAV"].iloc[-1], test1["NAV"].iloc[-1] * (1 - 0.001), atol=1e-12)

# test 4: 50/50 drifts, so monthly rebalancing trades a little
test4 = backtest_engine(half_half, close_fake, open_fake, returns_fake, eligible_fake, rebal_fake, cost_zero, cap=None, lookback=1)
assert (test4["Turnover"].iloc[1:] > 0).all() # every rebalance trades something
assert (test4["Turnover"].iloc[1:] < 0.05).all() # but only a little

# test 5: 100% AAA never drifts, so nothing to trade after the first trade
assert (test1["Turnover"].iloc[1:] < 1e-12).all()

print("backtest_engine checks passed")

backtest_engine checks passed


We will then create a performance estimator

In [69]:
# a function to evaluate performance of the allocator based on backtest engine's results
# wil return a dictionary with sharpe and max drowdown, plus data points to get there

def performance_calculator (nav, rf_daily):

    # calculate daily returns
    daily_returns = nav.pct_change().dropna()

    # annualised return
    total_growth = (nav.iloc[-1]/nav.iloc[0])
    n_years = len(daily_returns) / 252
    annualised_returns = total_growth**(1/n_years)-1

    # annualised volatility
    annualised_volatility = daily_returns.std()*np.sqrt(252)

    # Sharpe Ratio
    rf_alligned = rf_daily.reindex(daily_returns.index).fillna(0)
    excess_returns = daily_returns - rf_alligned
    sharpe = excess_returns.mean() / excess_returns.std() * np.sqrt(252)

    # max drawdown
    drawdown = nav / nav.cummax() - 1
    max_drawdown = drawdown.min()

    return pd.Series({
        "ann_return":   annualised_returns,
        "ann_vol":      annualised_volatility,
        "sharpe":       sharpe,
        "max_drawdown": max_drawdown,
    })

In [131]:
# function testing

dates = pd.bdate_range("2020-1-1", periods =253)
rf_zero = pd.Series(0, index=dates) # 0 daily
rf_positive = pd.Series(0.0001,index=dates) # 2.5% growth a yaer
nav_1=pd.Series(2**(np.arange(253)/252), index=dates) # smoothly growing NAV to 2x over a year
nav_2=pd.Series([1,1.2,0.9,1.1],index=dates[:4]) # four days with severe swings

test1 = performance_calculator(nav_1,rf_zero)
test2 = performance_calculator(nav_2, rf_zero)
test3 = performance_calculator(nav_1,rf_positive)
pd.DataFrame({"test1": test1, "test2": test2, "test3": test3})

,test1,test2,test3
ann_return,1.000000e+00,2998.062754,1.000000e+00
ann_vol,1.243739e-15,4.229832,1.243739e-15
sharpe,5.580762e+14,3.420151,5.378147e+14
max_drawdown,0.000000e+00,-0.250000,0.000000e+00
